# # Chicago Taxi Trips — Week of 11–17 September 2023

*Summary placeholder — written after the analysis is complete.*

# ## 1. Data loading
#
# All figures describe **reported** trips (the City notes that not every trip is reported), not total market volume.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

os.makedirs("figures/analysis", exist_ok=True)

# Chart styling: one blue for magnitude, orange to flag weekends, muted grey for context
BLUE, ORANGE, GREY = "#2a78d6", "#eb6834", "#9a9a95"
plt.rcParams.update({
    "figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": "#e6e6e3", "grid.linewidth": 0.8,
    "axes.axisbelow": True, "axes.titlesize": 13, "axes.titleweight": "bold",
    "axes.titlelocation": "left", "axes.labelsize": 10.5,
})

raw = pd.read_csv("trips.csv", dtype={"pickup_census_tract": "string", "dropoff_census_tract": "string"})
print("shape:", raw.shape)
print()
print(raw.dtypes)

shape: (135768, 23)

trip_id                           str
taxi_id                           str
trip_start_timestamp              str
trip_end_timestamp                str
trip_seconds                  float64
trip_miles                    float64
pickup_census_tract            string
dropoff_census_tract           string
pickup_community_area         float64
dropoff_community_area        float64
fare                          float64
tips                          float64
tolls                         float64
extras                        float64
trip_total                    float64
payment_type                      str
company                           str
pickup_centroid_latitude      float64
pickup_centroid_longitude     float64
pickup_centroid_location          str
dropoff_centroid_latitude     float64
dropoff_centroid_longitude    float64
dropoff_centroid_location         str
dtype: object


In [2]:
# Missing values per column, and uniqueness of trip_id
miss = raw.isna().sum().to_frame("n_missing")
miss["pct_missing"] = miss["n_missing"] / len(raw) * 100
print(miss[miss.n_missing > 0])
print()
print("duplicate trip_id:", raw.trip_id.duplicated().sum(), "| fully duplicated rows:", raw.duplicated().sum())
print("start range:", raw.trip_start_timestamp.min(), "->", raw.trip_start_timestamp.max())

                            n_missing  pct_missing
trip_end_timestamp                  1         0.00
trip_seconds                       33         0.02
pickup_census_tract             71404        52.59
dropoff_census_tract            73842        54.39
pickup_community_area            3815         2.81
dropoff_community_area          13794        10.16
fare                              324         0.24
tips                              324         0.24
tolls                             324         0.24
extras                            324         0.24
trip_total                        324         0.24
pickup_centroid_latitude         3775         2.78
pickup_centroid_longitude        3775         2.78
pickup_centroid_location         3775         2.78
dropoff_centroid_latitude       12872         9.48
dropoff_centroid_longitude      12872         9.48
dropoff_centroid_location       12872         9.48

duplicate trip_id: 0 | fully duplicated rows: 0
start range: 2023-09-11T00:00:00.